# Recreate the Hippo projection and reproduce Figures 3–4

**Run All on Sherlock in the original `sc-dl` environment, within a high-memory compute allocation.** This notebook starts from the surviving combined expression object at:

```text
/oak/stanford/groups/engreitz/Users/tri/shared/20260325_Hippo_timecourse/Hippo_timecourse_combined.h5ad
```

It does **not** read the wiped scratch directory and does **not** attempt to reconstruct the eight processed sublibraries. It validates the existing combined H5AD against the recovered production cohort, recreates the final selected top-specific-200 fixed-reference cNMF projection, validates and reconnects both inputs, runs the main-figure preflight, and then executes **Figure 3 (b–d, f)** and **Figure 4 (c–e)** in fresh kernels using the same Python interpreter.

## Input and generated artifact

| File | Role | Historical check |
|---|---|---|
| `Hippo_timecourse_combined.h5ad` | Existing read-only expression query | 212,891 cells × 62,757 features; all 54 sample counts; raw expression and required layers |
| `Hippo_timecourse_reprojected_cNMF_compare_gene_sets.cell_x_program.h5ad` | Recreated fixed-reference top-specific-200 usage and CLR | 212,891 cells × 50 programs; 7,268 selected genes |

The later condition-specific QC is recorded as a flag in the projected object (**99,416 passing cells**); it does not remove rows from either object. The figure helpers apply that flag and exclude sublibrary `s8` as in the recovered panel workflow.

The combined-query definitions come from production `Hippo_timecourse.ipynb` cells 5, 14, 20, 23, and 27 (zero based), preserved in `recovery/raw_downloads/Hippo_timecourse.ipynb`. The selected projection comes from `recovery/raw_downloads/reproject_cnmf_compare_gene_sets.ipynb` cells 3, 5, 7, 9, 11, and 13. Scientific validation and projection code is in `src/cc_perturbseq/hippo_rebuild.py` and `hippo_projection.py`.

See [the workflow notes](../../docs/HIPPO_REBUILD.md) for the original producer provenance and projection definitions.


In [ ]:
# Repository-path bootstrap for local support modules.
from pathlib import Path
import sys


def _find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "cc_perturbseq").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root containing "
        "src/cc_perturbseq. Run this notebook from within the "
        "cc-perturbseq-manuscript-analysis checkout."
    )


REPO_ROOT = _find_repo_root()
for _support_path in (REPO_ROOT, REPO_ROOT / "src"):
    _support_path_text = str(_support_path)
    if _support_path_text not in sys.path:
        sys.path.insert(0, _support_path_text)

print(
    "Repository support paths added:",
    REPO_ROOT,
    "and",
    REPO_ROOT / "src",
)


## 1. Settings

`QUERY_H5AD` points directly to the durable Oak file you located. The source object is read only; no copy is made unless you deliberately change the path. `REBUILD_ROOT` is still used for the newly generated projection, run records, configuration backups, and completed figure copies.

Keep `OVERWRITE = False` for normal use: a completed projection is reused only when its recorded provenance and validation match. Set it to `True` only to deliberately replace a previous projection. Keep `STRICT_HISTORICAL = True` for manuscript reproduction; a cohort mismatch needs investigation.

By default the final stage executes both affected figure notebooks. Set `RUN_FIGURE_NOTEBOOKS = False` if this run should stop after recreating and reconnecting the projection. The 2.1-million-cell concatenation stage has been removed, but fixed-reference projection of 212,891 cells is still memory intensive.


In [ ]:
from datetime import datetime, timezone
import json
import tempfile
from IPython.display import display

from src.cc_perturbseq.figure_notebook import get_path, load_paths
from src.cc_perturbseq.hippo_workflow import (
    check_runtime, execute_figure_notebooks, run_main_preflight,
    validate_hippo_panel_inputs, write_hippo_overrides,
)

config_file, paths = load_paths(REPO_ROOT)
REBUILD_ROOT = get_path(paths, "hippo_rebuild.output_root")
SPECTRA_TPM = get_path(paths, "hippo_rebuild.gene_spectra_tpm")

# Existing durable combined query discovered on Oak. This replaces the deleted
# eight-sublibrary scratch tree as the starting point for this notebook.
QUERY_H5AD = Path(
    "/oak/stanford/groups/engreitz/Users/tri/shared/"
    "20260325_Hippo_timecourse/Hippo_timecourse_combined.h5ad"
)

OVERWRITE = False
STRICT_HISTORICAL = True
COMPRESSION = None  # Projection output only; optionally "gzip" or "lzf".
RUN_FIGURE_NOTEBOOKS = True
FIGURE_NOTEBOOKS = ("Figure_03.ipynb", "Figure_04.ipynb")

PROJECTED_H5AD = (
    REBUILD_ROOT / "perturbseq_cNMF_projection" / "reprojection_gene_set_comparison"
    / "Hippo_timecourse_reprojected_cNMF_compare_gene_sets.cell_x_program.h5ad"
)
RUN_DIR = REBUILD_ROOT / "runs" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")

display({
    "configuration": str(config_file),
    "existing_query_input": str(QUERY_H5AD),
    "reference_spectra": str(SPECTRA_TPM),
    "projection_output": str(PROJECTED_H5AD),
    "run_record": str(RUN_DIR),
    "execute_figures": RUN_FIGURE_NOTEBOOKS,
})


## 2. Check dependencies, the existing query, and the fixed reference

This stage checks the durable combined H5AD instead of looking for the deleted sublibrary inputs. It also verifies the projection reference is the **gene-spectra TPM** file, not the similarly named consensus spectra used by program-annotation figures. Only the selected top-specific-200 mode is fitted; the HVG list and annotation workbook are not projection inputs. The workbook remains required by the main figure notebooks.


In [ ]:
versions = check_runtime(include_figures=RUN_FIGURE_NOTEBOOKS)
from src.cc_perturbseq.hippo_rebuild import validate_combined
from src.cc_perturbseq.hippo_projection import (
    project_top_specific_200, read_spectra, validate_projected,
)

if not QUERY_H5AD.is_file() or QUERY_H5AD.stat().st_size == 0:
    raise FileNotFoundError(f"Existing combined Hippo H5AD is missing or empty: {QUERY_H5AD}")

query_stat = QUERY_H5AD.stat()
query_record = {
    "path": str(QUERY_H5AD.resolve()),
    "size_bytes": int(query_stat.st_size),
    "mtime_ns": int(query_stat.st_mtime_ns),
    "role": "existing read-only combined expression query",
}

reference = read_spectra(SPECTRA_TPM, strict_historical=STRICT_HISTORICAL)
print("Reference shape:", reference.shape)
del reference

REBUILD_ROOT.mkdir(parents=True, exist_ok=True)
with tempfile.TemporaryFile(dir=REBUILD_ROOT) as probe:
    probe.write(b"Hippo output write check")

RUN_DIR.mkdir(parents=True, exist_ok=False)
settings_record = {
    "query_input": query_record,
    "spectra_tpm": str(SPECTRA_TPM),
    "projected_h5ad": str(PROJECTED_H5AD),
    "strict_historical": STRICT_HISTORICAL,
    "overwrite_projection": OVERWRITE,
    "projection_compression": COMPRESSION,
    "versions": versions,
}
(RUN_DIR / "settings.json").write_text(json.dumps(settings_record, indent=2) + "\n")
display(query_record)
display(versions)


## 3. Validate the existing expression query

The Oak H5AD is used directly and remains read only. Validation checks the recovered production shape and per-sample counts, the 54-sample mapping, broad-QC constraints, raw-count row sums, cell and feature identifiers, required metadata, count layers, and guide matrices.

This replaces the old concatenation cell. No deleted scratch input is accessed.


In [ ]:
query_validation = validate_combined(
    QUERY_H5AD,
    strict_historical=STRICT_HISTORICAL,
)
combined_report = {
    "status": "existing_combined_validated",
    "rebuilt": False,
    "input_file": query_record,
    "validation": query_validation,
}
(RUN_DIR / "combined_stage.json").write_text(
    json.dumps(combined_report, indent=2) + "\n"
)
print("Existing combined query validated:", QUERY_H5AD)
display(query_validation)


## 4. Recreate the final selected fixed-reference projection

For each of the 50 programs, genes are ranked by their fraction of total reference weight, then by program weight. The first-occurrence union of each program's top 200 genes is intersected with query symbols. Duplicate query symbols are summed, and expression is normalized using the original full-library totals to 1,000,000 per cell.

The existing spectra are fixed. The source multiplicative-update solve uses float32, Frobenius loss, 1,000 maximum iterations, tolerance `1e-5`, and seed `0`. CLR uses normalized usage, floor `1e-6`, renormalization, and centered log2 values. The named matrix consumed by the figures is `obsm["X_cnmf_clr_top_specific_200"]`, with explicit columns `P0`…`P49`.

Only this selected mode is stored. In the generated file, `X` also holds selected normalized usage; downstream figures read the named CLR matrix. Iterations and any convergence warning are recorded; parameters are never changed automatically. The original selected solve reported 140 iterations, which is a comparison point rather than a required cross-version numerical identity.


In [ ]:
projection_report = project_top_specific_200(
    QUERY_H5AD, SPECTRA_TPM, PROJECTED_H5AD, overwrite=OVERWRITE,
    strict_historical=STRICT_HISTORICAL, compression=COMPRESSION,
)
(RUN_DIR / "projection_stage.json").write_text(
    json.dumps(projection_report, indent=2) + "\n"
)
display(projection_report["diagnostics"])
display(projection_report["validation"])


## 5. Validate the pair and reconnect the figure notebooks

Validation checks complete cell identity/order, metadata, the original usage-to-CLR calculation, program labels, and historical counts. The figures additionally require their marker genes and every requested condition after QC. Local configuration is updated only after pair validation; unrelated settings are preserved and the previous YAML is backed up under this run directory.

The query override is set to the existing Oak combined H5AD, while the projection override points to the newly generated top-specific-200 object.


In [ ]:
pair_validation = validate_projected(
    QUERY_H5AD, PROJECTED_H5AD, strict_historical=STRICT_HISTORICAL,
)
panel_validation = validate_hippo_panel_inputs(QUERY_H5AD)
config_update = write_hippo_overrides(
    REPO_ROOT, QUERY_H5AD, PROJECTED_H5AD, backup_dir=RUN_DIR / "config_backups",
)
(RUN_DIR / "pair_validation.json").write_text(
    json.dumps(pair_validation, indent=2) + "\n"
)
(RUN_DIR / "panel_validation.json").write_text(
    json.dumps(panel_validation, indent=2) + "\n"
)
(RUN_DIR / "configuration_update.json").write_text(
    json.dumps(config_update, indent=2) + "\n"
)
display(pair_validation)
display(panel_validation)
display(config_update)


## 6. Preflight and execute Figures 3 and 4

The existing main-figure preflight checks all four main notebooks and their configured inputs. When it succeeds, the two affected notebooks run sequentially in clean kernels using **this Python environment**, with no execution time limit. Each notebook saves its normal figures/source tables; executed notebook copies, failure traces, a run receipt, and copies of successful figure directories are kept under the Oak run directory.

If a downstream notebook fails, the validated input and generated projection remain available. Fix the reported error and rerun this final cell, or open the relevant main notebook directly. Successful execution does not by itself establish agreement with the assembled manuscript; inspect the generated figures and source tables.


In [ ]:
if RUN_FIGURE_NOTEBOOKS:
    preflight_report = run_main_preflight(
        REPO_ROOT, RUN_DIR / "main_figure_preflight.log"
    )
    figure_report = execute_figure_notebooks(
        REPO_ROOT, RUN_DIR, notebook_names=FIGURE_NOTEBOOKS,
    )
    display(figure_report)
else:
    print(
        "The existing query and generated projection are configured. "
        "Figure execution was disabled in settings."
    )

print("Expression query (existing Oak input):", QUERY_H5AD)
print("Selected projection (generated):", PROJECTED_H5AD)
print("Run records and completed figure copies:", RUN_DIR)
